In [8]:
import pandas as pd
import emoji

from collections import Counter

# Leitura do datasets Parquet
train_path = "../../../00-source/olid-br/data/train-00000-of-00001-0d8933b80051ca0e.parquet"
test_path = "../../../00-source/olid-br/data/test-00000-of-00001-914dbee7561d2266.parquet"

# Carregando os dados
train0_df = pd.read_parquet(train_path)
test0_df = pd.read_parquet(test_path)

# Concatenando os dados
train_df = pd.concat(
    [train0_df, test0_df],
    ignore_index=True
)

In [9]:
# Extração dos emojis dos textos
def extrair_emojis(texto):
    if pd.isna(texto):
        return []
    return [char for char in texto if char in emoji.EMOJI_DATA]

train_df["emojis"] = train_df["text"].apply(extrair_emojis)

# Identificação das mensagens de ódio
df_odio = train_df[
    train_df["is_offensive"] == "OFF"
] # OFF = ofensivo

df_nao_odio = train_df[
    train_df["is_offensive"] == "NOT"
] # NOT = não ofensivo

# Conta quantas mensagens ofensivas possuem cada emoji
emojis_odio = Counter(
    e for conjunto in df_odio["emojis"] for e in conjunto
)

# Conta quantas mensagens não ofensivas possuem cada emoji
emojis_nao_odio = Counter(
    e for conjunto in df_nao_odio["emojis"] for e in conjunto
)

# Junta todos os emojis encontrados nas duas categorias
todos = set(emojis_odio) | set(emojis_nao_odio)

# Montagem da tabela final
df_emojis = pd.DataFrame({
    "emoji": list(todos),
    "freq_odio": [emojis_odio.get(e, 0) for e in todos],
    "freq_nao_odio": [
        emojis_nao_odio.get(e, 0)
        for e in todos
    ],
})

df_emojis["freq_total"] = (
    df_emojis["freq_odio"] +
    df_emojis["freq_nao_odio"]
)

# Normalização das frequências
total_odio = df_odio.shape[0]
total_nao_odio = df_nao_odio.shape[0]

df_emojis["freq_odio_norm"] = (
    df_emojis["freq_odio"] / total_odio
)

df_emojis["freq_nao_odio_norm"] = (
    df_emojis["freq_nao_odio"] / total_nao_odio
)

# Salvamento da tabela original
df_emojis.to_csv(
    "youtube_emojis_olid-br.csv",
    index=False,
    encoding="utf-8"
)

# Salvamento da tabela normalizada
df_emojis.to_csv(
    "youtube_emojis_olid-br_normalizado.csv",
    index=False,
    encoding="utf-8"
)